In [0]:
# ================================================================
# CELL 1 — IMPORTS
# ================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

print("=" * 70)
print("DATABRICKS JOB CREATION")
print("=" * 70)

w = WorkspaceClient()

print("Databricks SDK initialized.")

In [0]:
# ================================================================
# CELL 2 — JOB CONFIGURATION
# ================================================================

PROJECT_ROOT = (
    "/Users/vnvarkhede@gmail.com/"
    "databricks-genai-data-analyst-copilot"
)

PIPELINE_ROOT = (
    PROJECT_ROOT +
    "/notebooks/09_pipeline"
)

JOB_NAME = "genai-data-analyst-copilot-pipeline"

DATA_PIPELINE = (
    PIPELINE_ROOT +
    "/02_data_pipeline.py"
)

RAG_PIPELINE = (
    PIPELINE_ROOT +
    "/03_rag_pipeline.py"
)

COPILOT_PIPELINE = (
    PIPELINE_ROOT +
    "/04_copilot_pipeline.py"
)

print("Job name:")
print(JOB_NAME)

print()
print("Project root:")
print(PROJECT_ROOT)

print()
print("Pipeline root:")
print(PIPELINE_ROOT)

print()
print("Data pipeline:")
print(DATA_PIPELINE)

print()
print("RAG pipeline:")
print(RAG_PIPELINE)

print()
print("Copilot pipeline:")
print(COPILOT_PIPELINE)

In [0]:
# ================================================================
# CELL 3 — VERIFY PIPELINE FILES
# ================================================================

print("=" * 70)
print("PIPELINE FILE VALIDATION")
print("=" * 70)

pipeline_files = {
    "DATA": DATA_PIPELINE,
    "RAG": RAG_PIPELINE,
    "COPILOT": COPILOT_PIPELINE
}

missing_files = []

for name, path in pipeline_files.items():

    try:

        w.workspace.get_status(
            path
        )

        print(
            f"PASS - {name}: {path}"
        )

    except Exception as e:

        print(
            f"FAIL - {name}: {path}"
        )

        print(
            "      ",
            type(e).__name__,
            str(e)
        )

        missing_files.append(
            name
        )

print()
print(
    "Total files:",
    len(pipeline_files)
)

print(
    "Missing files:",
    len(missing_files)
)

if missing_files:

    raise RuntimeError(
        "Missing pipeline files: "
        + ", ".join(missing_files)
    )

print()
print("Pipeline file validation: PASS")

In [0]:
# ================================================================
# CELL 4 — CHECK EXISTING JOB
# ================================================================

print("=" * 70)
print("CHECKING EXISTING JOBS")
print("=" * 70)

existing_job_id = None

for job in w.jobs.list(
    name=JOB_NAME
):

    print(
        "Found existing job:"
    )

    print(
        "Job ID:",
        job.job_id
    )

    print(
        "Name:",
        job.settings.name
        if job.settings
        else JOB_NAME
    )

    existing_job_id = job.job_id

if existing_job_id is None:

    print(
        "No existing job found."
    )

else:

    print()
    print(
        "Existing Job ID:",
        existing_job_id
    )

In [0]:
# ================================================================
# CELL 5 — SERVERLESS COMPUTE CONFIGURATION
# ================================================================

print("=" * 70)
print("SERVERLESS COMPUTE CONFIGURATION")
print("=" * 70)

# This project uses Databricks Serverless.
# No cluster ID or Spark version is required.

USE_SERVERLESS = True

print(
    "Compute type:",
    "SERVERLESS"
)

print(
    "Cluster ID:",
    "Not required"
)

print(
    "Spark version:",
    "Managed by Databricks"
)

print()
print("Serverless configuration: PASS")

In [0]:
# ================================================================
# CELL 6 — DEFINE SERVERLESS PIPELINE TASKS
# ================================================================

print("=" * 70)
print("SERVERLESS JOB TASKS")
print("=" * 70)

tasks = [

    jobs.Task(
        task_key="data_pipeline",

        spark_python_task=jobs.SparkPythonTask(
            python_file=DATA_PIPELINE
        ),

        environment_key="default"
    ),

    jobs.Task(
        task_key="rag_pipeline",

        depends_on=[
            jobs.TaskDependency(
                task_key="data_pipeline"
            )
        ],

        spark_python_task=jobs.SparkPythonTask(
            python_file=RAG_PIPELINE
        ),

        environment_key="default"
    ),

    jobs.Task(
        task_key="copilot_pipeline",

        depends_on=[
            jobs.TaskDependency(
                task_key="rag_pipeline"
            )
        ],

        spark_python_task=jobs.SparkPythonTask(
            python_file=COPILOT_PIPELINE
        ),

        environment_key="default"
    )
]

for task in tasks:

    print(
        f"PASS - {task.task_key}"
    )

print()
print(
    "Total tasks:",
    len(tasks)
)

print(
    "Serverless task configuration: PASS"
)

In [0]:
# ================================================================
# CELL 7 — SERVERLESS JOB ENVIRONMENT
# ================================================================

print("=" * 70)
print("SERVERLESS JOB ENVIRONMENT")
print("=" * 70)

from databricks.sdk.service import compute

environment = jobs.JobEnvironment(
    environment_key="default",

    spec=compute.Environment(
        environment_version="2"
    )
)

print(
    "Environment key:",
    "default"
)

print(
    "Environment version:",
    "2"
)

print(
    "Compute:",
    "Serverless"
)

print()
print("Serverless environment: PASS")

In [0]:
# ================================================================
# CELL 9 — UPDATE EXISTING JOB TO SERVERLESS
# ================================================================

print("=" * 70)
print("UPDATING EXISTING JOB")
print("=" * 70)

w.jobs.reset(
    job_id=job_id,

    new_settings=jobs.JobSettings(
        name=JOB_NAME,
        tasks=tasks,
        environments=[environment],
        max_concurrent_runs=1
    )
)

print(
    "Job updated successfully."
)

print(
    "Job ID:",
    job_id
)

print(
    "Compute:",
    "SERVERLESS"
)

print()
print("Job update: PASS")

In [0]:
# ================================================================
# CELL 10 — VERIFY SERVERLESS JOB
# ================================================================

print("=" * 70)
print("VERIFYING SERVERLESS JOB")
print("=" * 70)

job_info = w.jobs.get(
    job_id=job_id
)

print(
    "Job ID:",
    job_info.job_id
)

print(
    "Job name:",
    job_info.settings.name
)

print()

for task in job_info.settings.tasks:

    print(
        "TASK:",
        task.task_key
    )

    print(
        "Python file:",
        task.spark_python_task.python_file
    )

    print(
        "Environment:",
        task.environment_key
    )

    print()

print(
    "Environments:",
    len(
        job_info.settings.environments
        or []
    )
)

print()
print("Serverless Job verification: PASS")

In [0]:
# ================================================================
# CELL 11 — VALIDATE JOB PIPELINE PATHS
# ================================================================

print("=" * 70)
print("VALIDATING JOB PIPELINE PATHS")
print("=" * 70)

expected_paths = {
    "data_pipeline": DATA_PIPELINE,
    "rag_pipeline": RAG_PIPELINE,
    "copilot_pipeline": COPILOT_PIPELINE
}

path_errors = []

for task in job_info.settings.tasks:

    expected = expected_paths.get(
        task.task_key
    )

    actual = (
        task.spark_python_task.python_file
        if task.spark_python_task
        else None
    )

    passed = (
        actual == expected
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{task.task_key}"
    )

    print(
        "  Expected:",
        expected
    )

    print(
        "  Actual:",
        actual
    )

    if not passed:

        path_errors.append(
            task.task_key
        )

print()

if path_errors:

    raise RuntimeError(
        "Incorrect Python file paths: "
        + ", ".join(path_errors)
    )

print(
    "Pipeline path validation: PASS"
)

In [0]:
# ================================================================
# CELL 12 — VALIDATE TASK DEPENDENCIES
# ================================================================

print("=" * 70)
print("VALIDATING TASK DEPENDENCIES")
print("=" * 70)

expected_dependencies = {
    "data_pipeline": [],
    "rag_pipeline": [
        "data_pipeline"
    ],
    "copilot_pipeline": [
        "rag_pipeline"
    ]
}

dependency_errors = []

for task in job_info.settings.tasks:

    actual = [
        dependency.task_key
        for dependency in (
            task.depends_on or []
        )
    ]

    expected = expected_dependencies.get(
        task.task_key,
        []
    )

    passed = (
        actual == expected
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{task.task_key}"
    )

    print(
        "  Expected:",
        expected
    )

    print(
        "  Actual:",
        actual
    )

    if not passed:

        dependency_errors.append(
            task.task_key
        )

print()

if dependency_errors:

    raise RuntimeError(
        "Incorrect dependencies: "
        + ", ".join(dependency_errors)
    )

print(
    "Dependency validation: PASS"
)

In [0]:
# ================================================================
# CELL 13 — RUN SERVERLESS JOB
# ================================================================

print("=" * 70)
print("STARTING SERVERLESS JOB")
print("=" * 70)

run_response = w.jobs.run_now(
    job_id=job_id
)

run_id = run_response.run_id

print(
    "Job ID:",
    job_id
)

print(
    "Run ID:",
    run_id
)

print()
print(
    "Serverless Job started: PASS"
)